# Aequitas — Data Audit Notebook

**Purpose:** Profile every raw government data source before writing a single line of pipeline code.
Lock in ground truth entity counts. Document data traps. Establish join key relationships.

**Rule:** Every number written into `GROUND_TRUTH` at the end of this notebook is final.
Pipeline code will be validated against these numbers. If the pipeline produces different numbers, the pipeline is wrong.

## Sources covered
1. NaPTAN — bus stops
2. BODS — bus routes (9 regional feeds)
3. ONS Census 2021 — LSOA population + demographics
4. IMD 2019 — deprivation scores
5. NOMIS — unemployment (MSOA level)
6. ONS GeoJSON Boundaries — LSOA + region polygons

## Output
`data/audit/ground_truth.json` — locked counts, referenced by all downstream validation gates.

# 0. Setup

In [ ]:
import sys
import json
import zipfile
import io
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = Path("..").resolve()
RAW        = ROOT / "data" / "raw"
AUDIT_DIR  = ROOT / "data" / "audit"

NAPTAN_DIR    = RAW / "naptan"
BODS_DIR      = RAW / "bods"
CENSUS_DIR    = RAW / "census"
IMD_DIR       = RAW / "imd"
NOMIS_DIR     = RAW / "nomis"
BOUNDARY_DIR  = RAW / "boundaries"

for d in [NAPTAN_DIR, BODS_DIR, CENSUS_DIR, IMD_DIR, NOMIS_DIR, BOUNDARY_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Helpers ────────────────────────────────────────────────────────────────
def section(title: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def profile_df(df: pd.DataFrame, name: str) -> None:
    """Print a concise profile of a DataFrame."""
    print(f"\n── {name} ──")
    print(f"  Rows:    {len(df):,}")
    print(f"  Columns: {list(df.columns)}")
    print(f"\n  Null counts (non-zero only):")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print("    None")
    else:
        for col, n in nulls.items():
            print(f"    {col}: {n:,} ({n/len(df)*100:.1f}%)")
    print(f"\n  Sample (3 rows):")
    print(df.head(3).to_string())

# Ground truth accumulator — filled throughout the notebook
GT: dict = {
    "generated_at": datetime.utcnow().isoformat(),
    "naptan": {},
    "bods": {},
    "census": {},
    "imd": {},
    "nomis": {},
    "boundaries": {},
    "joins": {},
}

print("Setup complete.")
print(f"ROOT: {ROOT}")
print(f"Python: {sys.version}")

# 1. NaPTAN — Bus Stops

**Source:** https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv  
**What we expect:** ~700k+ rows total (all UK transport nodes). We filter to England bus stops only.  
**Key trap:** Includes rail, tram, ferry, taxi ranks. Must filter `StopType` to `BCT`, `BCS`, `BCE` only.

In [ ]:
section("1. NaPTAN — Download")

NAPTAN_CSV = NAPTAN_DIR / "Stops.csv"

if not NAPTAN_CSV.exists():
    print("Downloading NaPTAN...")
    url = "https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv"
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    NAPTAN_CSV.write_bytes(resp.content)
    print(f"Saved: {NAPTAN_CSV} ({NAPTAN_CSV.stat().st_size / 1_000_000:.1f} MB)")
else:
    print(f"Already exists: {NAPTAN_CSV} ({NAPTAN_CSV.stat().st_size / 1_000_000:.1f} MB)")

In [ ]:
section("1. NaPTAN — Profile raw file")

naptan_raw = pd.read_csv(NAPTAN_CSV, low_memory=False)
profile_df(naptan_raw, "NaPTAN raw")

print(f"\n  StopType value counts:")
print(naptan_raw["StopType"].value_counts().to_string())

In [ ]:
section("1. NaPTAN — Filter to England bus stops")

# Step 1: filter to bus stop types only
BUS_STOP_TYPES = {"BCT", "BCS", "BCE"}
naptan_bus = naptan_raw[naptan_raw["StopType"].isin(BUS_STOP_TYPES)].copy()
print(f"  After StopType filter: {len(naptan_bus):,} rows")

# Step 2: drop inactive stops
if "Status" in naptan_bus.columns:
    status_counts = naptan_bus["Status"].value_counts()
    print(f"\n  Status value counts:\n{status_counts.to_string()}")
    naptan_bus = naptan_bus[naptan_bus["Status"] == "active"].copy()
    print(f"\n  After active-only filter: {len(naptan_bus):,} rows")

# Step 3: filter to England by bounding box
# England: lat 49.9–55.8, lon –6.4 to 2.0
# (excludes Scotland, Wales, NI which are outside BODS scope)
LAT_COL = "Latitude"
LON_COL = "Longitude"

naptan_bus[LAT_COL] = pd.to_numeric(naptan_bus[LAT_COL], errors="coerce")
naptan_bus[LON_COL] = pd.to_numeric(naptan_bus[LON_COL], errors="coerce")

invalid_coords = naptan_bus[LAT_COL].isna() | naptan_bus[LON_COL].isna()
print(f"\n  Rows with invalid coords: {invalid_coords.sum():,}")

naptan_england = naptan_bus[
    naptan_bus[LAT_COL].between(49.9, 55.8) &
    naptan_bus[LON_COL].between(-6.4, 2.0) &
    ~invalid_coords
].copy()
print(f"  After England bounds filter: {len(naptan_england):,} rows")

# Step 4: check ATCO code uniqueness
atco_col = "ATCOCode"
if atco_col in naptan_england.columns:
    total = len(naptan_england)
    unique_atco = naptan_england[atco_col].nunique()
    dupes = total - unique_atco
    print(f"\n  Total rows: {total:,}")
    print(f"  Unique ATCOCodes: {unique_atco:,}")
    print(f"  Duplicate ATCOCodes: {dupes:,}")
    if dupes > 0:
        print("  ⚠ DUPLICATES FOUND — investigate before pipeline")
        dupe_sample = naptan_england[naptan_england.duplicated(atco_col, keep=False)].head(6)
        print(dupe_sample[[atco_col, "StopType", "CommonName", LAT_COL, LON_COL]].to_string())

# Lock ground truth
GT["naptan"] = {
    "raw_total_rows": int(len(naptan_raw)),
    "bus_type_rows": int(len(naptan_bus)),
    "england_active_bus_stops": int(len(naptan_england)),
    "unique_atco_codes": int(naptan_england[atco_col].nunique()) if atco_col in naptan_england.columns else None,
    "has_duplicates": bool(dupes > 0) if atco_col in naptan_england.columns else None,
}
print(f"\n  ✓ NaPTAN ground truth locked: {GT['naptan']}")

# 2. BODS — Bus Routes

**Source:** https://data.bus-data.dft.gov.uk/timetable/download/gtfs-file/all/  
**What we expect:** 9 regional GTFS zip files. Each contains `routes.txt`, `trips.txt`, `stops.txt`, `stop_times.txt`.  
**Key traps:**
- `trips.txt` rows ≠ routes. One route → many trips. Count from `routes.txt` only.
- Same route can appear in multiple regional feeds. Deduplicate on `route_id` across all regions.
- `route_id` format varies by operator — inspect before assuming uniqueness.

In [ ]:
section("2. BODS — Download all regional GTFS feeds")

# BODS provides one combined GTFS for all of England
# Individual operator feeds exist but the combined is the right starting point
BODS_GTFS_ZIP = BODS_DIR / "bods_gtfs_all.zip"

if not BODS_GTFS_ZIP.exists():
    print("Downloading BODS GTFS (all England)...")
    url = "https://data.bus-data.dft.gov.uk/timetable/download/gtfs-file/all/"
    resp = requests.get(url, timeout=300, stream=True)
    resp.raise_for_status()
    with open(BODS_GTFS_ZIP, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved: {BODS_GTFS_ZIP} ({BODS_GTFS_ZIP.stat().st_size / 1_000_000:.1f} MB)")
else:
    print(f"Already exists: {BODS_GTFS_ZIP} ({BODS_GTFS_ZIP.stat().st_size / 1_000_000:.1f} MB)")

# List contents
with zipfile.ZipFile(BODS_GTFS_ZIP) as zf:
    print(f"\n  Files in zip:")
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"    {name} ({info.file_size / 1_000_000:.1f} MB uncompressed)")

In [ ]:
section("2. BODS — Profile routes.txt and trips.txt")

with zipfile.ZipFile(BODS_GTFS_ZIP) as zf:
    with zf.open("routes.txt") as f:
        bods_routes = pd.read_csv(f, low_memory=False)
    with zf.open("trips.txt") as f:
        bods_trips = pd.read_csv(f, low_memory=False)
    with zf.open("stops.txt") as f:
        bods_stops = pd.read_csv(f, low_memory=False)

profile_df(bods_routes, "BODS routes.txt")
print(f"\n  route_type value counts (3=bus, 0=tram, 2=rail):")
print(bods_routes["route_type"].value_counts().to_string())

print(f"\n── BODS trips.txt ──")
print(f"  Rows: {len(bods_trips):,}  (this is NOT route count)")
print(f"  Unique route_ids in trips: {bods_trips['route_id'].nunique():,}")

print(f"\n── BODS stops.txt ──")
print(f"  Rows: {len(bods_stops):,}")
print(f"  Unique stop_ids: {bods_stops['stop_id'].nunique():,}")

In [ ]:
section("2. BODS — Deduplicate and count unique bus routes")

# Filter to bus routes only (route_type == 3)
bods_bus_routes = bods_routes[bods_routes["route_type"] == 3].copy()
print(f"  Bus routes (route_type=3): {len(bods_bus_routes):,}")

# Check route_id uniqueness
total_route_rows = len(bods_bus_routes)
unique_route_ids = bods_bus_routes["route_id"].nunique()
duplicate_route_rows = total_route_rows - unique_route_ids

print(f"  Unique route_ids: {unique_route_ids:,}")
print(f"  Duplicate route_id rows: {duplicate_route_rows:,}")

if duplicate_route_rows > 0:
    print("\n  ⚠ Duplicate route_ids found — sample:")
    dupes = bods_bus_routes[bods_bus_routes.duplicated("route_id", keep=False)]
    print(dupes[["route_id", "route_short_name", "route_long_name", "agency_id"]].head(8).to_string())

# Check BODS stops vs NaPTAN stops overlap
# BODS stop_ids should be ATCO codes — verify format
print(f"\n  BODS stop_id sample (should be ATCO codes):")
print(bods_stops["stop_id"].head(10).to_string())

# How many BODS stops match NaPTAN ATCOCodes?
if "ATCOCode" in naptan_england.columns:
    naptan_atco_set = set(naptan_england["ATCOCode"].astype(str))
    bods_stop_set = set(bods_stops["stop_id"].astype(str))
    overlap = naptan_atco_set & bods_stop_set
    bods_only = bods_stop_set - naptan_atco_set
    naptan_only = naptan_atco_set - bods_stop_set
    print(f"\n  NaPTAN England stops: {len(naptan_atco_set):,}")
    print(f"  BODS stops: {len(bods_stop_set):,}")
    print(f"  Overlap (in both): {len(overlap):,}")
    print(f"  BODS stops not in NaPTAN: {len(bods_only):,}")
    print(f"  NaPTAN stops not in BODS: {len(naptan_only):,}")

# Lock ground truth
GT["bods"] = {
    "raw_route_rows": int(total_route_rows),
    "unique_bus_route_ids": int(unique_route_ids),
    "total_trips": int(len(bods_trips)),
    "bods_stops_total": int(len(bods_stops)),
    "bods_unique_stop_ids": int(bods_stops["stop_id"].nunique()),
    "bods_naptan_stop_overlap": int(len(overlap)) if "ATCOCode" in naptan_england.columns else None,
}
print(f"\n  ✓ BODS ground truth locked: {GT['bods']}")

# 3. ONS Census 2021 — LSOA Population & Demographics

**Source:** NOMIS API — bulk download of Census 2021 TS001 (population) and TS011 (car ownership), TS007 (age), TS021 (ethnicity)  
**What we expect:** 33,755 LSOAs in England. Population sum ~56.3M.  
**Key trap:** Census files come at different geographic levels. Always verify LSOA-level sum matches ONS England total.

In [ ]:
section("3. Census — Download LSOA population (TS001)")

# NOMIS bulk download for Census 2021 TS001 (Usual resident population) at LSOA level
# Geography: 2021 LSOAs in England (type 139)
CENSUS_POP_CSV = CENSUS_DIR / "census2021_ts001_lsoa_population.csv"

if not CENSUS_POP_CSV.exists():
    print("Downloading Census 2021 TS001 (population by LSOA)...")
    # NOMIS API: dataset=NM_2028_1 = Census 2021 TS001
    # geography=TYPE139 = 2021 LSOAs
    url = (
        "https://www.nomisweb.co.uk/api/v01/dataset/NM_2028_1.csv"
        "?geography=TYPE139"
        "&c2021_relpop_6=0"
        "&measures=20100"
        "&select=geography_code,geography_name,obs_value"
    )
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    CENSUS_POP_CSV.write_bytes(resp.content)
    print(f"Saved: {CENSUS_POP_CSV} ({CENSUS_POP_CSV.stat().st_size / 1_000:.0f} KB)")
else:
    print(f"Already exists: {CENSUS_POP_CSV}")

In [ ]:
section("3. Census — Profile and validate population totals")

census_pop = pd.read_csv(CENSUS_POP_CSV)
profile_df(census_pop, "Census 2021 TS001 — LSOA population")

# Rename for clarity
census_pop.columns = [c.lower().strip() for c in census_pop.columns]
# Expected cols: geography_code, geography_name, obs_value
pop_col = [c for c in census_pop.columns if "obs_value" in c or "value" in c][0]
code_col = [c for c in census_pop.columns if "code" in c][0]

# Filter to England only (LSOA codes start with E)
census_england = census_pop[census_pop[code_col].str.startswith("E", na=False)].copy()
print(f"\n  England LSOAs: {len(census_england):,}")

total_population = census_england[pop_col].sum()
ONS_ENGLAND_POPULATION = 56_490_048  # ONS Census 2021 official England total
discrepancy = abs(total_population - ONS_ENGLAND_POPULATION)
discrepancy_pct = discrepancy / ONS_ENGLAND_POPULATION * 100

print(f"\n  Sum of LSOA populations: {total_population:,.0f}")
print(f"  ONS official England total: {ONS_ENGLAND_POPULATION:,}")
print(f"  Discrepancy: {discrepancy:,.0f} ({discrepancy_pct:.2f}%)")

if discrepancy_pct > 0.5:
    print("  ⚠ DISCREPANCY > 0.5% — wrong geography level or filtered file")
else:
    print("  ✓ Population totals match within tolerance")

print(f"\n  Population stats:")
print(f"    Min LSOA population: {census_england[pop_col].min():,}")
print(f"    Max LSOA population: {census_england[pop_col].max():,}")
print(f"    Mean LSOA population: {census_england[pop_col].mean():.0f}")
print(f"    LSOAs with pop = 0: {(census_england[pop_col] == 0).sum():,}")

GT["census"] = {
    "total_lsoas_england": int(len(census_england)),
    "total_population_sum": int(total_population),
    "ons_official_england_population": ONS_ENGLAND_POPULATION,
    "population_discrepancy_pct": round(discrepancy_pct, 4),
    "lsoa_code_column": code_col,
    "population_column": pop_col,
}
print(f"\n  ✓ Census ground truth locked: {GT['census']}")

# 4. IMD 2019 — Deprivation Scores

**Source:** https://assets.publishing.service.gov.uk/media/5d8b3b51ed915d03720bab73/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv  
**What we expect:** 32,844 LSOAs (2015 boundaries). ~900 fewer than Census 2021's 33,755.  
**Key trap:** IMD uses 2015 LSOA boundaries. Census 2021 uses 2021 boundaries. Must document how to handle the ~900 boundary-changed LSOAs.

In [ ]:
section("4. IMD 2019 — Download and profile")

IMD_CSV = IMD_DIR / "imd2019_scores_ranks_deciles.csv"

if not IMD_CSV.exists():
    print("Downloading IMD 2019...")
    url = (
        "https://assets.publishing.service.gov.uk/media/"
        "5d8b3b51ed915d03720bab73/"
        "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    IMD_CSV.write_bytes(resp.content)
    print(f"Saved: {IMD_CSV} ({IMD_CSV.stat().st_size / 1_000:.0f} KB)")
else:
    print(f"Already exists: {IMD_CSV}")

imd = pd.read_csv(IMD_CSV)
profile_df(imd, "IMD 2019")

# Identify the LSOA code column
print(f"\n  Column names:")
for col in imd.columns:
    print(f"    {col}")

In [ ]:
section("4. IMD 2019 — Boundary mismatch analysis")

# Identify LSOA code column (usually 'LSOA code (2011)')
lsoa_col_imd = [c for c in imd.columns if "lsoa" in c.lower() and "code" in c.lower()][0]
print(f"  LSOA code column: '{lsoa_col_imd}'")

imd_lsoa_codes = set(imd[lsoa_col_imd].dropna().astype(str))
census_lsoa_codes = set(census_england[code_col].dropna().astype(str))

in_imd_not_census = imd_lsoa_codes - census_lsoa_codes
in_census_not_imd = census_lsoa_codes - imd_lsoa_codes

print(f"\n  IMD LSOAs: {len(imd_lsoa_codes):,}")
print(f"  Census 2021 LSOAs: {len(census_lsoa_codes):,}")
print(f"  In IMD but not Census 2021: {len(in_imd_not_census):,}")
print(f"  In Census 2021 but not IMD: {len(in_census_not_imd):,}  ← these have no deprivation score")

print(f"\n  Sample Census 2021 LSOAs with no IMD score:")
print(list(in_census_not_imd)[:10])

# IMD score distribution
imd_score_col = [c for c in imd.columns if "score" in c.lower() and "imd" in c.lower()][0]
imd_decile_col = [c for c in imd.columns if "decile" in c.lower() and "imd" in c.lower()][0]
print(f"\n  IMD score column: '{imd_score_col}'")
print(f"  IMD decile column: '{imd_decile_col}'")
print(f"\n  Decile distribution:")
print(imd[imd_decile_col].value_counts().sort_index().to_string())

GT["imd"] = {
    "total_lsoas": int(len(imd_lsoa_codes)),
    "lsoas_no_imd_score": int(len(in_census_not_imd)),
    "lsoa_code_column": lsoa_col_imd,
    "imd_score_column": imd_score_col,
    "imd_decile_column": imd_decile_col,
    "boundary_note": "IMD 2019 uses 2011 LSOA boundaries. ~900 Census 2021 LSOAs have no IMD score. Strategy: assign IMD score from spatially overlapping 2011 LSOA using ONS lookup table.",
}
print(f"\n  ✓ IMD ground truth locked")

# 5. NOMIS — Unemployment (MSOA level)

**Source:** NOMIS API — Annual Population Survey, unemployment by MSOA  
**What we expect:** ~7,000 MSOAs in England.  
**Key trap:** Published at MSOA level, not LSOA. Must distribute to LSOA using ONS MSOA→LSOA lookup. All LSOAs within an MSOA share the same unemployment rate.

In [ ]:
section("5. NOMIS — Download MSOA unemployment")

NOMIS_CSV = NOMIS_DIR / "nomis_unemployment_msoa.csv"

if not NOMIS_CSV.exists():
    print("Downloading NOMIS unemployment by MSOA...")
    # NM_17_5 = Annual Population Survey — economic activity
    # geography TYPE265 = 2021 MSOAs in England
    url = (
        "https://www.nomisweb.co.uk/api/v01/dataset/NM_17_5.csv"
        "?geography=TYPE265"
        "&variable=84"       # unemployment rate
        "&measures=20100"
        "&select=geography_code,geography_name,obs_value"
    )
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    NOMIS_CSV.write_bytes(resp.content)
    print(f"Saved: {NOMIS_CSV} ({NOMIS_CSV.stat().st_size / 1_000:.0f} KB)")
else:
    print(f"Already exists: {NOMIS_CSV}")

nomis = pd.read_csv(NOMIS_CSV)
nomis.columns = [c.lower().strip() for c in nomis.columns]
profile_df(nomis, "NOMIS unemployment by MSOA")

msoa_code_col = [c for c in nomis.columns if "code" in c][0]
unemp_col = [c for c in nomis.columns if "obs_value" in c or "value" in c][0]

msoa_england = nomis[nomis[msoa_code_col].str.startswith("E", na=False)].copy()
print(f"\n  England MSOAs: {len(msoa_england):,}")
print(f"  MSOAs with null unemployment: {msoa_england[unemp_col].isna().sum():,}")
print(f"  Unemployment rate stats:")
print(f"    Min: {msoa_england[unemp_col].min():.1f}%")
print(f"    Max: {msoa_england[unemp_col].max():.1f}%")
print(f"    Mean: {msoa_england[unemp_col].mean():.1f}%")

GT["nomis"] = {
    "total_msoas_england": int(len(msoa_england)),
    "msoas_with_null_unemployment": int(msoa_england[unemp_col].isna().sum()),
    "msoa_code_column": msoa_code_col,
    "unemployment_column": unemp_col,
    "distribution_note": "MSOA unemployment distributed to constituent LSOAs via ONS lookup. All LSOAs in same MSOA share the same rate.",
}
print(f"\n  ✓ NOMIS ground truth locked")

# 6. ONS GeoJSON Boundaries

**Sources:**
- LSOA boundaries (2021): ONS Geography portal
- Region boundaries (2021): ONS Geography portal

**Purpose:** Point-in-polygon assignment of bus stops → LSOA, and LSOA → region.  
**Key check:** Boundary files must cover 100% of England. No gaps.

In [ ]:
section("6. Boundaries — Download LSOA and Region GeoJSON")

import json as json_lib

LSOA_GEOJSON   = BOUNDARY_DIR / "lsoa_2021_england_buc.geojson"
REGION_GEOJSON = BOUNDARY_DIR / "regions_2021_england_buc.geojson"

# LSOA 2021 boundaries (BUC = Boundary Ultra Clipped — smaller file, good enough for point-in-polygon)
if not LSOA_GEOJSON.exists():
    print("Downloading LSOA 2021 boundaries...")
    url = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_2021_EW_BUC_V3/FeatureServer/0/query"
        "?where=LSOA21CD+LIKE+'E%25'"
        "&outFields=LSOA21CD,LSOA21NM"
        "&f=geojson"
        "&resultRecordCount=40000"
    )
    resp = requests.get(url, timeout=180)
    resp.raise_for_status()
    LSOA_GEOJSON.write_bytes(resp.content)
    print(f"Saved: {LSOA_GEOJSON} ({LSOA_GEOJSON.stat().st_size / 1_000_000:.1f} MB)")
else:
    print(f"Already exists: {LSOA_GEOJSON} ({LSOA_GEOJSON.stat().st_size / 1_000_000:.1f} MB)")

# Region boundaries
if not REGION_GEOJSON.exists():
    print("Downloading Region boundaries...")
    url = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "Regions_December_2021_EN_BUC/FeatureServer/0/query"
        "?where=1%3D1"
        "&outFields=RGN21CD,RGN21NM"
        "&f=geojson"
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    REGION_GEOJSON.write_bytes(resp.content)
    print(f"Saved: {REGION_GEOJSON}")
else:
    print(f"Already exists: {REGION_GEOJSON}")

In [ ]:
section("6. Boundaries — Profile and validate coverage")

with open(LSOA_GEOJSON) as f:
    lsoa_geo = json_lib.load(f)

with open(REGION_GEOJSON) as f:
    region_geo = json_lib.load(f)

lsoa_features = lsoa_geo["features"]
region_features = region_geo["features"]

print(f"  LSOA features: {len(lsoa_features):,}")
print(f"  Region features: {len(region_features):,}")

# Verify LSOA codes match Census
geo_lsoa_codes = {f["properties"].get("LSOA21CD") for f in lsoa_features}
geo_lsoa_codes.discard(None)

geo_only = geo_lsoa_codes - census_lsoa_codes
census_only = census_lsoa_codes - geo_lsoa_codes

print(f"\n  GeoJSON LSOAs: {len(geo_lsoa_codes):,}")
print(f"  Census LSOAs: {len(census_lsoa_codes):,}")
print(f"  In GeoJSON but not Census: {len(geo_only):,}")
print(f"  In Census but not GeoJSON: {len(census_only):,}")
if census_only:
    print(f"  ⚠ {len(census_only):,} LSOAs have no boundary — point-in-polygon will miss these")

# Region codes
region_codes = {f["properties"].get("RGN21CD") for f in region_features}
print(f"\n  Regions in GeoJSON: {sorted(region_codes)}")

EXPECTED_REGIONS = {
    "E12000001", "E12000002", "E12000003", "E12000004", "E12000005",
    "E12000006", "E12000007", "E12000008", "E12000009"
}
missing_regions = EXPECTED_REGIONS - region_codes
if missing_regions:
    print(f"  ⚠ Missing regions: {missing_regions}")
else:
    print(f"  ✓ All 9 England regions present")

GT["boundaries"] = {
    "lsoa_features": len(lsoa_features),
    "region_features": len(region_features),
    "lsoas_missing_boundary": len(census_only),
    "all_9_regions_present": len(missing_regions) == 0,
}
print(f"\n  ✓ Boundaries ground truth locked")

# 7. Join Audit — Stop → LSOA Match Rate

This is the most critical join in the entire pipeline. Every per-capita metric depends on it.  
We do a spatial point-in-polygon test on a sample to estimate match rate before committing to full pipeline.

In [ ]:
section("7. Join Audit — Stop → LSOA spatial match (sample)")

# Use shapely for point-in-polygon on a 1,000 stop sample
# If shapely not installed: pip install shapely
try:
    from shapely.geometry import shape, Point
    SHAPELY_AVAILABLE = True
except ImportError:
    SHAPELY_AVAILABLE = False
    print("  ⚠ shapely not installed. Run: pip install shapely")
    print("  Skipping spatial match — install shapely and re-run this cell.")

if SHAPELY_AVAILABLE:
    print("  Building LSOA polygon index from GeoJSON...")
    lsoa_polygons = []
    for feature in lsoa_features:
        try:
            geom = shape(feature["geometry"])
            code = feature["properties"].get("LSOA21CD")
            lsoa_polygons.append((code, geom))
        except Exception:
            continue
    print(f"  Built {len(lsoa_polygons):,} LSOA polygons")

    # Sample 1,000 bus stops
    sample = naptan_england.sample(min(1000, len(naptan_england)), random_state=42)
    matched = 0
    unmatched_coords = []

    print(f"  Testing {len(sample):,} stop sample...")
    for _, row in sample.iterrows():
        pt = Point(row[LON_COL], row[LAT_COL])
        found = False
        for code, poly in lsoa_polygons:
            if poly.contains(pt):
                matched += 1
                found = True
                break
        if not found:
            unmatched_coords.append((row[LAT_COL], row[LON_COL]))

    match_rate = matched / len(sample) * 100
    print(f"\n  Matched: {matched:,} / {len(sample):,} ({match_rate:.1f}%)")

    if match_rate < 95:
        print(f"  ⚠ Match rate below 95% — investigate before building pipeline")
        print(f"  Sample unmatched coords (first 5): {unmatched_coords[:5]}")
    else:
        print(f"  ✓ Match rate acceptable (>{95}%)")

    GT["joins"] = {
        "stop_to_lsoa_sample_size": len(sample),
        "stop_to_lsoa_sample_match_rate_pct": round(match_rate, 2),
        "note": "Full match rate computed during pipeline processing stage.",
    }
else:
    GT["joins"] = {"note": "shapely not available — spatial match not tested in audit"}

print(f"\n  ✓ Join audit complete")

# 8. Lock Ground Truth

Write `ground_truth.json`. This file is the contract for the entire pipeline.  
**Do not edit this file manually after locking. If numbers change, re-run the audit.**

In [ ]:
section("8. Lock Ground Truth")

GT_PATH = AUDIT_DIR / "ground_truth.json"
GT["locked_at"] = datetime.utcnow().isoformat()

with open(GT_PATH, "w") as f:
    json_lib.dump(GT, f, indent=2)

print(f"  Ground truth written to: {GT_PATH}")
print(f"\n  Summary:")
print(f"    NaPTAN England active bus stops: {GT['naptan'].get('england_active_bus_stops', 'NOT SET'):,}" if isinstance(GT['naptan'].get('england_active_bus_stops'), int) else f"    NaPTAN: NOT SET")
print(f"    BODS unique bus routes: {GT['bods'].get('unique_bus_route_ids', 'NOT SET'):,}" if isinstance(GT['bods'].get('unique_bus_route_ids'), int) else f"    BODS: NOT SET")
print(f"    Census 2021 England LSOAs: {GT['census'].get('total_lsoas_england', 'NOT SET'):,}" if isinstance(GT['census'].get('total_lsoas_england'), int) else f"    Census: NOT SET")
print(f"    Census population sum: {GT['census'].get('total_population_sum', 'NOT SET'):,}" if isinstance(GT['census'].get('total_population_sum'), int) else f"    Census pop: NOT SET")
print(f"    IMD 2019 LSOAs: {GT['imd'].get('total_lsoas', 'NOT SET'):,}" if isinstance(GT['imd'].get('total_lsoas'), int) else f"    IMD: NOT SET")
print(f"    LSOAs with no IMD score: {GT['imd'].get('lsoas_no_imd_score', 'NOT SET'):,}" if isinstance(GT['imd'].get('lsoas_no_imd_score'), int) else f"    IMD mismatch: NOT SET")
print(f"    NOMIS England MSOAs: {GT['nomis'].get('total_msoas_england', 'NOT SET'):,}" if isinstance(GT['nomis'].get('total_msoas_england'), int) else f"    NOMIS: NOT SET")
print(f"    Stop→LSOA match rate: {GT['joins'].get('stop_to_lsoa_sample_match_rate_pct', 'NOT SET')}%" if GT['joins'].get('stop_to_lsoa_sample_match_rate_pct') else f"    Join match: NOT SET")

print(f"\n  ✓ Audit complete. Ground truth locked.")
print(f"  Next step: build src/aequitas/core/models.py using these confirmed entity definitions.")